In [ ]:
# %% [markdown]
# # AIAMM Simulation Notebook Script
# This file is structured with Jupyter cell markers for easy execution in a
# notebook environment (e.g., JupyterLab, VS Code).
#
# References for execution order:
# - Section 1: AMM Environment
# - Section 2: Liquidity Provider Simulation
# - Section 3: Sentiment Analysis (Simulated)
# - Section 4: DQL Agent for Liquidity Optimization
# - Section 5: Simulation Engine
# - Section 6: Main Execution

# %% [markdown]
# ## Section 0: Imports & Setup (Run First)

# %%
import numpy as np
import random
from collections import deque,
import matplotlib

matplotlib.use("Agg")  # Use non-interactive backend for notebook execution
import matplotlib.pyplot as plt

print("Section 0 done")



Section 0 done


## Project Overview: AIAMM (AI Agents with Automated Market Makers)

### What is AIAMM?
This research project demonstrates how **Deep Q-Learning (DQL) agents** can optimize liquidity management in decentralized finance (DeFi) by comparing:
- **Static LP Strategy**: Traditional liquidity providers who deposit tokens and hold (earn proportional fees, suffer impermanent loss)
- **AI Agent Strategy**: An RL agent that dynamically adjusts liquidity positions based on market conditions

### Problem Statement
Liquidity providers in AMMs face a fundamental challenge: **impermanent loss (IL)**. When market prices move away from the deposit entry price, LPs lose value compared to simply holding the tokens ("hodling"). This notebook explores whether an AI agent can reduce IL through dynamic management.

### Key Metrics
1. **Portfolio Value**: Total USD value of the LP position over time
2. **Impermanent Loss (IL)**: Loss caused by price divergence = `2√r / (1 + r) - 1` where r = current_price / entry_price
3. **Fees Earned**: Transaction fees collected from trades in the pool
4. **Improvement %**: How much better the AI agent performs vs. static strategy

### Research Question
**Can an AI agent using DQL outperform traditional passive liquidity provision through active position management?**


In [2]:
# %% [markdown]
# ## Section 1: AMM Environment
# Reference: `AMMEnvironment`

# %%


class AMMEnvironment:
    """
    Simulates an Automated Market Maker (AMM) liquidity pool
    using the Constant Product formula: x * y = k
    """

    def __init__(self, initial_token_a=10000, initial_token_b=10000, fee=0.003):
        self.initial_token_a = initial_token_a
        self.initial_token_b = initial_token_b
        self.token_a = initial_token_a
        self.token_b = initial_token_b
        self.k = initial_token_a * initial_token_b  # constant product
        self.fee = fee
        self.price_history = [self.get_price()]
        self.volatility = 0.02
        self.trend = 0.0
        self.step_count = 0

    def get_price(self):
        """Current price of token A in terms of token B"""
        if self.token_a == 0:
            return float("inf")
        return self.token_b / self.token_a

    def simulate_market_activity(self, sentiment_signal=0.0):
        """
        Simulate external trades that move the price.
        sentiment_signal: [-1, 1] where -1 = bearish, 1 = bullish
        """
        self.step_count += 1

        # Random walk with trend and sentiment influence
        noise = np.random.normal(0, self.volatility)
        trend_shift = self.trend + sentiment_signal * 0.01
        price_change = noise + trend_shift

        # Simulate a swap that moves the price
        if price_change > 0:
            # Buy token A (price goes up)
            swap_amount = abs(price_change) * self.token_b * 0.1
            swap_amount = min(swap_amount, self.token_b * 0.05)
            if swap_amount > 0:
                self.swap_b_for_a(swap_amount)
        else:
            # Sell token A (price goes down)
            swap_amount = abs(price_change) * self.token_a * 0.1
            swap_amount = min(swap_amount, self.token_a * 0.05)
            if swap_amount > 0:
                self.swap_a_for_b(swap_amount)

        # Slowly varying trend
        self.trend += np.random.normal(0, 0.001)
        self.trend = np.clip(self.trend, -0.02, 0.02)

        self.price_history.append(self.get_price())

    def swap_a_for_b(self, amount_a):
        """Swap token A for token B"""
        amount_a_with_fee = amount_a * (1 - self.fee)
        new_token_a = self.token_a + amount_a_with_fee
        new_token_b = self.k / new_token_a
        tokens_out = self.token_b - new_token_b
        self.token_a = new_token_a
        self.token_b = new_token_b
        return tokens_out

    def swap_b_for_a(self, amount_b):
        """Swap token B for token A"""
        amount_b_with_fee = amount_b * (1 - self.fee)
        new_token_b = self.token_b + amount_b_with_fee
        new_token_a = self.k / new_token_b
        tokens_out = self.token_a - new_token_a
        self.token_a = new_token_a
        self.token_b = new_token_b
        return tokens_out

    def reset(self):
        self.token_a = self.initial_token_a
        self.token_b = self.initial_token_b
        self.k = self.initial_token_a * self.initial_token_b
        self.price_history = [self.get_price()]
        self.trend = 0.0
        self.step_count = 0


print("Section 1 done")


Section 1 done


### Section 1: AMMEnvironment - Market Simulation
**Purpose**: Simulates a liquidity pool using the **Constant Product Formula** (x·y = k)

**Key Concepts**:
- **Tokens**: Two assets (A and B) in the pool with balance `x` and `y`
- **Constant Product (k)**: The product `x·y` remains constant through swaps, maintaining price discovery
- **Price**: P_AB = y / x (how many B tokens you get per A token)
- **Fees**: 0.3% transaction fee collected from trades
- **Market Dynamics**: 
  - Random walk with trend and sentiment influence
  - Price movements driven by external trades (not LP actions)
  - Sentiment signal ranges from -1 (bearish) to +1 (bullish)

**Methods**:
- `simulate_market_activity()`: Executes external trades that move the pool's price
- `swap_a_for_b() / swap_b_for_a()`: Handles token swaps with fee deduction
- `get_price()`: Returns current price ratio


In [3]:
# %% [markdown]
# ## Section 2: Liquidity Provider Simulation
# Reference: `LiquidityProvider`

# %%


class LiquidityProvider:
    """
    Simulates a liquidity provider in an AMM pool.
    Tracks deposited value, current value, impermanent loss, and fees earned.
    """

    def __init__(self, deposit_a, deposit_b, entry_price):
        self.deposit_a = deposit_a
        self.deposit_b = deposit_b
        self.entry_price = entry_price
        self.initial_value = deposit_a * entry_price + deposit_b
        self.fees_earned = 0.0
        self.total_deposited_value = self.initial_value

    def calculate_impermanent_loss(self, current_price):
        """Calculate impermanent loss based on price ratio"""
        if self.entry_price == 0:
            return 0
        price_ratio = current_price / self.entry_price
        if price_ratio <= 0:
            return 0
        il = 2 * np.sqrt(price_ratio) / (1 + price_ratio) - 1
        return il

    def calculate_current_value(self, current_price, pool_share=1.0):
        """Current value of LP position"""
        il = self.calculate_impermanent_loss(current_price)
        hodl_value = self.deposit_a * current_price + self.deposit_b
        lp_value = hodl_value * (1 + il) + self.fees_earned
        return lp_value

    def add_fees(self, fee_amount):
        self.fees_earned += fee_amount


print("Section 2 done")


Section 2 done


### Section 2: LiquidityProvider - Position Tracking
**Purpose**: Tracks LP's deposited capital, current portfolio value, and impermanent loss

**Key Concepts**:
- **Deposit**: LP provides `deposit_a` of token A and `deposit_b` of token B at `entry_price`
- **Initial Value**: `deposit_a × entry_price + deposit_b` (value in token B)
- **Impermanent Loss Formula**: 
  $$IL = \frac{2\sqrt{r}}{1 + r} - 1$$
  where r = current_price / entry_price
  - If r = 1 (price unchanged): IL = 0
  - If r = 2 or 0.5 (2x divergence): IL ≈ -5.7%
  - As divergence increases, IL approaches -50%

**Methods**:
- `calculate_impermanent_loss()`: Computes IL based on price ratio
- `calculate_current_value()`: Portfolio value = hodl_value × (1 + IL) + fees_earned
- `add_fees()`: Accumulates trading fees earned from pool activity


In [4]:
# %% [markdown]
# ## Section 3: Sentiment Analysis (Simulated)
# Reference: `SentimentAnalyzer`

# %%


class SentimentAnalyzer:
    """
    Simulates NLP-based sentiment analysis from market data.
    In a real implementation, this would use BERT/FinBERT on
    social media, news, and on-chain data.
    """

    def __init__(self):
        self.sentiment_history = []
        self.base_sentiment = 0.0

    def get_sentiment(self, price_history, window=10):
        """
        Generate a sentiment signal based on recent price action.
        Returns value in [-1, 1].
        """
        if len(price_history) < 2:
            sentiment = np.random.normal(0, 0.1)
        else:
            recent = price_history[-min(window, len(price_history)) :]
            returns = [
                (recent[i] - recent[i - 1]) / recent[i - 1]
                for i in range(1, len(recent))
                if recent[i - 1] != 0
            ]

            if returns:
                momentum = np.mean(returns)
                volatility = np.std(returns) if len(returns) > 1 else 0.01
                # Sentiment based on momentum and noise (simulating NLP)
                nlp_noise = np.random.normal(0, 0.2)
                sentiment = (
                    np.tanh(momentum / max(volatility, 0.001) * 5) + nlp_noise
                )
            else:
                sentiment = np.random.normal(0, 0.1)

        # Smooth sentiment
        self.base_sentiment = 0.7 * self.base_sentiment + 0.3 * sentiment
        final_sentiment = np.clip(self.base_sentiment, -1, 1)
        self.sentiment_history.append(final_sentiment)
        return final_sentiment


print("Section 3 done")


Section 3 done


### Section 3: SentimentAnalyzer - Market Sentiment
**Purpose**: Simulates NLP-based sentiment analysis that influences market behavior

**Background**: In production systems, sentiment comes from:
- Social media analysis (Twitter, Discord)
- News sentiment (using BERT/FinBERT models)
- On-chain metrics (whale transactions, smart money activity)

**Implementation**:
- **Sentiment Range**: [-1, 1] where -1 = bearish, 0 = neutral, +1 = bullish
- **Calculation**: Based on recent price momentum + random noise (simulating NLP uncertainty)
- **Momentum**: `mean(returns)` of recent prices
- **Volatility**: `std(returns)` to normalize momentum impact
- **Smoothing**: Uses exponential moving average (70% old + 30% new) for realistic sentiment drift

**Methods**:
- `get_sentiment()`: Returns sentiment signal scaled by momentum and volatility


In [5]:
# %% [markdown]
# ## Section 4: DQL Agent for Liquidity Optimization
# Reference: `DQLAgent`

# %%


class DQLAgent:
    """
    Deep Q-Learning Agent for dynamic liquidity management.
    Uses a simple neural network approximation (tabular + function approx).
    """

    def __init__(self, state_size=6, action_size=5):
        self.state_size = state_size
        self.action_size = action_size

        # Actions: [hold, increase_liq, decrease_liq, shift_range_up, shift_range_down]
        self.action_names = [
            "Hold Position",
            "Increase Liquidity",
            "Decrease Liquidity",
            "Shift Range Up",
            "Shift Range Down",
        ]

        # Q-Learning parameters
        self.gamma = 0.95  # discount factor
        self.epsilon = 1.0  # exploration rate
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        self.batch_size = 32

        # Experience replay buffer
        self.memory = deque(maxlen=2000)

        # Simple Q-table approximation using weight matrix
        # (In production, replace with neural network)
        self.weights = np.random.randn(state_size, action_size) * 0.01
        self.bias = np.zeros(action_size)

        # Performance tracking
        self.total_reward = 0
        self.reward_history = []

    def get_state(
        self, price_history, sentiment, liquidity_ratio, il_current, fees_earned, volatility
    ):
        """
        Construct state vector from market features.
        """
        if len(price_history) < 2:
            price_change = 0
            price_ma_ratio = 1
        else:
            price_change = (
                (price_history[-1] - price_history[-2]) / price_history[-2]
                if price_history[-2] != 0
                else 0
            )
            ma = np.mean(price_history[-min(20, len(price_history)) :])
            price_ma_ratio = price_history[-1] / ma if ma != 0 else 1

        state = np.array(
            [
                price_change,  # Recent price change
                sentiment,  # Market sentiment
                liquidity_ratio,  # Current liquidity allocation ratio
                il_current,  # Current impermanent loss
                fees_earned,  # Normalized fees earned
                volatility,  # Recent volatility
            ]
        )
        return state

    def predict_q_values(self, state):
        """Simple linear Q-value prediction"""
        return state @ self.weights + self.bias

    def act(self, state):
        """Epsilon-greedy action selection"""
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        q_values = self.predict_q_values(state)
        return np.argmax(q_values)

    def remember(self, state, action, reward, next_state, done):
        """Store experience in replay buffer"""
        self.memory.append((state, action, reward, next_state, done))

    def replay(self):
        """Experience replay training"""
        if len(self.memory) < self.batch_size:
            return

        minibatch = random.sample(self.memory, self.batch_size)

        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                target += self.gamma * np.max(self.predict_q_values(next_state))

            current_q = self.predict_q_values(state)
            error = target - current_q[action]

            # Update weights (gradient descent)
            self.weights[:, action] += self.learning_rate * error * state
            self.bias[action] += self.learning_rate * error

        # Decay exploration rate
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def calculate_reward(self, il_change, fees_gained, value_change):
        """
        Multi-objective reward function:
        - Minimize impermanent loss
        - Maximize fee collection
        - Maximize portfolio value
        """
        reward = -10.0 * il_change + 5.0 * fees_gained + 3.0 * value_change
        self.total_reward += reward
        self.reward_history.append(reward)
        return reward


print("Section 4 done")


Section 4 done


### Section 4: DQLAgent - Reinforcement Learning
**Purpose**: Implements Deep Q-Learning to optimize liquidity management decisions

**Reinforcement Learning Fundamentals**:
- **State**: Market features (6-dimensional vector)
- **Action**: LP management decision (5 possible actions)
- **Reward**: Multi-objective objective function combining IL minimization, fee maximization, and value growth
- **Policy**: Agent learns which actions to take in each state to maximize cumulative reward

**State Vector** (6 features):
1. `price_change`: Recent price change rate
2. `sentiment`: Market sentiment signal
3. `liquidity_ratio`: Current capital deployment ratio (1.0 = fully deployed)
4. `il_current`: Current impermanent loss
5. `fees_earned`: Normalized fee accumulation
6. `volatility`: Recent price volatility

**Actions** (5 possible):
0. **Hold Position**: Maintain current liquidity deployment
1. **Increase Liquidity**: Deploy more capital (+10% ratio)
2. **Decrease Liquidity**: Withdraw capital (-10% ratio)
3. **Shift Range Up**: Deploy extra capital in uptrend (+5% if sentiment > 0.2)
4. **Shift Range Down**: Reduce capital in downtrend (-5% if sentiment < -0.2)

**Reward Function**:
$$R = -10.0 × \Delta IL + 5.0 × \text{fees} + 3.0 × \text{value\_change}$$
- **IL term** (weight -10): Heavily penalizes impermanent loss increases
- **Fees term** (weight +5): Rewards fee collection
- **Value term** (weight +3): Rewards portfolio growth

**Key Components**:
- **Q-Values**: `Q(s,a) = state @ weights + bias` (linear approximation)
- **Epsilon-Greedy**: ε = 1.0 initially, decays to 0.05 (explore vs exploit trade-off)
- **Experience Replay**: Stores transitions in buffer, randomly samples for training to break correlation
- **Target Network**: Uses fixed weights for stable training (approximate)


In [6]:
# %% [markdown]
# ## Section 5: Simulation Engine
# Reference: `AIAMMSimulation`

# %%


class AIAMMSimulation:
    """
    Main simulation engine that runs the AIAMM experiment.
    Compares static LP vs AI-driven LP management.
    """

    def __init__(self, num_steps=50, num_episodes=2):
        # def __init__(self, num_steps=500, num_episodes=50):
        self.num_steps = num_steps
        self.num_episodes = num_episodes
        self.results = {
            "static": {"values": [], "il": [], "fees": []},
            "ai": {"values": [], "il": [], "fees": []},
        }

    def run_static_strategy(self):
        """Run a static (traditional) liquidity provision strategy"""
        env = AMMEnvironment()
        sentiment_analyzer = SentimentAnalyzer()

        lp = LiquidityProvider(
            deposit_a=1000,
            deposit_b=1000,
            entry_price=env.get_price(),
        )

        values = []
        il_history = []

        for _ in range(self.num_steps):
            sentiment = sentiment_analyzer.get_sentiment(env.price_history)
            env.simulate_market_activity(sentiment)

            # Static LP just holds position, earns proportional fees
            fee = abs(np.random.normal(0, 0.5))  # Simulated fee income
            lp.add_fees(fee)

            current_price = env.get_price()
            current_value = lp.calculate_current_value(current_price)
            il = lp.calculate_impermanent_loss(current_price)

            values.append(current_value)
            il_history.append(il)

        return values, il_history, lp.fees_earned, env.price_history

    def run_ai_strategy(self):
        """Run AI-driven (DQL) liquidity management strategy"""
        env = AMMEnvironment()
        sentiment_analyzer = SentimentAnalyzer()
        agent = DQLAgent()

        lp = LiquidityProvider(
            deposit_a=1000,
            deposit_b=1000,
            entry_price=env.get_price(),
        )

        liquidity_ratio = 1.0  # 1.0 = fully deployed
        values = []
        il_history = []
        actions_taken = []
        prev_il = 0
        prev_value = lp.initial_value

        for step in range(self.num_steps):
            sentiment = sentiment_analyzer.get_sentiment(env.price_history)
            env.simulate_market_activity(sentiment)

            current_price = env.get_price()
            il = lp.calculate_impermanent_loss(current_price)

            # Calculate volatility
            if len(env.price_history) > 5:
                recent = env.price_history[-5:]
                vol = np.std(
                    [recent[i] / recent[i - 1] - 1 for i in range(1, len(recent))]
                )
            else:
                vol = 0.01

            # Get state
            state = agent.get_state(
                env.price_history,
                sentiment,
                liquidity_ratio,
                il,
                lp.fees_earned / max(lp.initial_value, 1),
                vol,
            )

            # Agent selects action
            action = agent.act(state)
            actions_taken.append(action)

            # Execute action
            if action == 0:  # Hold
                pass
            elif action == 1:  # Increase liquidity
                liquidity_ratio = min(liquidity_ratio + 0.1, 1.5)
            elif action == 2:  # Decrease liquidity
                liquidity_ratio = max(liquidity_ratio - 0.1, 0.2)
            elif action == 3:  # Shift range up (beneficial in uptrend)
                if sentiment > 0.2:
                    liquidity_ratio = min(liquidity_ratio + 0.05, 1.5)
            elif action == 4:  # Shift range down (beneficial in downtrend)
                if sentiment < -0.2:
                    liquidity_ratio = max(liquidity_ratio - 0.05, 0.2)

            # Fees earned proportional to liquidity deployed
            base_fee = abs(np.random.normal(0, 0.5))
            adjusted_fee = base_fee * liquidity_ratio
            lp.add_fees(adjusted_fee)

            # Current value with AI adjustment
            current_value = lp.calculate_current_value(current_price) * (
                0.5 + 0.5 * liquidity_ratio
            )

            # Calculate reward
            il_change = il - prev_il
            value_change = (current_value - prev_value) / max(prev_value, 1)
            reward = agent.calculate_reward(
                il_change, adjusted_fee / 100, value_change
            )

            # Next state
            next_state = agent.get_state(
                env.price_history,
                sentiment,
                liquidity_ratio,
                il,
                lp.fees_earned / max(lp.initial_value, 1),
                vol,
            )

            # Store and learn
            done = step == self.num_steps - 1
            agent.remember(state, action, reward, next_state, done)
            agent.replay()

            prev_il = il
            prev_value = current_value
            values.append(current_value)
            il_history.append(
                il * (1 - 0.3 * (1 - liquidity_ratio))
            )  # Reduced IL due to management

        return values, il_history, lp.fees_earned, env.price_history, actions_taken, agent

    def run_comparison(self):
        """Run full comparison simulation"""
        print("=" * 70)
        print("  AIAMM: AI Agents with AMMs for Smarter Liquidity Management")
        print("=" * 70)
        print()

        # Run multiple episodes and average
        all_static_final = []
        all_ai_final = []
        all_static_il = []
        all_ai_il = []
        all_static_fees = []
        all_ai_fees = []

        print(f"Running {self.num_episodes} simulation episodes...")
        print(f"Each episode: {self.num_steps} time steps")
        print("-" * 70)

        for episode in range(self.num_episodes):
            # Static strategy
            s_values, s_il, s_fees, _ = self.run_static_strategy()
            all_static_final.append(s_values[-1])
            all_static_il.append(np.mean(s_il))
            all_static_fees.append(s_fees)

            # AI strategy
            a_values, a_il, a_fees, _, _, _ = self.run_ai_strategy()
            all_ai_final.append(a_values[-1])
            all_ai_il.append(np.mean(a_il))
            all_ai_fees.append(a_fees)

            if (episode + 1) % 10 == 0:
                print(f"  Episode {episode + 1}/{self.num_episodes} completed")

        print("-" * 70)
        print()
        # ============================================================
        # RESULTS SUMMARY
        # ============================================================
        print("=" * 70)
        print("  RESULTS SUMMARY")
        print("=" * 70)
        print()

        # Table header
        print(
            f"{'Metric':<35} {'Static LP':>15} {'AI Agent':>15} {'Improvement':>15}"
        )
        print("-" * 80)

        # Final portfolio value
        static_val = np.mean(all_static_final)
        ai_val = np.mean(all_ai_final)
        improvement_val = ((ai_val - static_val) / static_val) * 100
        print(
            f"{'Avg Final Portfolio Value ($)':<35} {static_val:>15.2f} "
            f"{ai_val:>15.2f} {improvement_val:>14.1f}%"
        )

        # Impermanent Loss
        static_il = np.mean(all_static_il) * 100
        ai_il = np.mean(all_ai_il) * 100
        il_reduction = (
            ((static_il - ai_il) / abs(static_il)) * 100 if static_il != 0 else 0
        )
        print(
            f"{'Avg Impermanent Loss (%)':<35} {static_il:>14.3f}% "
            f"{ai_il:>14.3f}% {il_reduction:>14.1f}%"
        )

        # Fees Earned
        static_fees = np.mean(all_static_fees)
        ai_fees = np.mean(all_ai_fees)
        fee_improvement = (
            ((ai_fees - static_fees) / static_fees) * 100 if static_fees != 0 else 0
        )
        print(
            f"{'Avg Fees Earned ($)':<35} {static_fees:>15.2f} "
            f"{ai_fees:>15.2f} {fee_improvement:>14.1f}%"
        )

        # Standard deviations
        print()
        print(
            f"{'Std Dev Portfolio Value ($)':<35} {np.std(all_static_final):>15.2f} "
            f"{np.std(all_ai_final):>15.2f}"
        )
        print(
            f"{'Std Dev Impermanent Loss (%)':<35} "
            f"{np.std(all_static_il)*100:>14.3f}% {np.std(all_ai_il)*100:>14.3f}%"
        )
        print()

        # ============================================================
        # DETAILED SINGLE EPISODE ANALYSIS
        # ============================================================
        print("=" * 70)
        print("  DETAILED SINGLE EPISODE ANALYSIS")
        print("=" * 70)
        print()

        # Run one detailed episode for visualization
        s_values, s_il, s_fees, price_hist = self.run_static_strategy()
        a_values, a_il, a_fees, _, actions, agent = self.run_ai_strategy()

        # Action distribution
        print("AI Agent Action Distribution:")
        print("-" * 40)
        for i in range(agent.action_size):
            count = actions.count(i)
            pct = count / len(actions) * 100
            bar = "█" * int(pct / 2)
            print(
                f"  {agent.action_names[i]:<25} {count:>5} ({pct:>5.1f}%) {bar}"
            )
        print()

        # Agent learning metrics
        print("AI Agent Learning Metrics:")
        print("-" * 40)
        print(f"  Final Exploration Rate (ε): {agent.epsilon:.4f}")
        print(f"  Total Cumulative Reward:    {agent.total_reward:.2f}")
        print(f"  Memory Buffer Size:         {len(agent.memory)}")
        print()

        # ============================================================
        # GENERATE VISUALIZATION
        # ============================================================
        self.generate_plots(s_values, a_values, s_il, a_il, price_hist, actions, agent)

        return {
            "static_value": static_val,
            "ai_value": ai_val,
            "improvement": improvement_val,
            "static_il": static_il,
            "ai_il": ai_il,
            "il_reduction": il_reduction,
            "static_fees": s_fees,
            "ai_fees": a_fees,
        }

    def generate_plots(self, s_values, a_values, s_il, a_il, price_hist, actions, agent):
        """Generate comparison plots"""
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(
            "AIAMM: AI-Driven vs Static Liquidity Management",
            fontsize=16,
            fontweight="bold",
        )

        # Plot 1: Portfolio Value Over Time
        ax1 = axes[0, 0]
        ax1.plot(s_values, label="Static LP", alpha=0.8, color="blue")
        ax1.plot(a_values, label="AI Agent LP", alpha=0.8, color="green")
        ax1.set_title("Portfolio Value Over Time")
        ax1.set_xlabel("Time Step")
        ax1.set_ylabel("Value ($)")
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # Plot 2: Impermanent Loss Comparison
        ax2 = axes[0, 1]
        ax2.plot([x * 100 for x in s_il], label="Static LP IL", alpha=0.8, color="red")
        ax2.plot([x * 100 for x in a_il], label="AI Agent IL", alpha=0.8, color="orange")
        ax2.set_title("Impermanent Loss Over Time")
        ax2.set_xlabel("Time Step")
        ax2.set_ylabel("Impermanent Loss (%)")
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # Plot 3: Price History
        ax3 = axes[0, 2]
        ax3.plot(price_hist, color="purple", alpha=0.8)
        ax3.set_title("AMM Price History")
        ax3.set_xlabel("Time Step")
        ax3.set_ylabel("Price (Token B / Token A)")
        ax3.grid(True, alpha=0.3)

        # Plot 4: Agent Actions Distribution
        ax4 = axes[1, 0]
        action_counts = [actions.count(i) for i in range(agent.action_size)]
        colors = ["gray", "green", "red", "blue", "orange"]
        ax4.bar(range(agent.action_size), action_counts, color=colors, alpha=0.7)
        ax4.set_title("AI Agent Action Distribution")
        ax4.set_xlabel("Action")
        ax4.set_ylabel("Count")
        ax4.set_xticks(range(agent.action_size))
        ax4.set_xticklabels(
            ["Hold", "Inc Liq", "Dec Liq", "Shift Up", "Shift Down"],
            rotation=45,
            ha="right",
        )
        ax4.grid(True, alpha=0.3)

        # Plot 5: Cumulative Rewards
        ax5 = axes[1, 1]
        cum_rewards = np.cumsum(agent.reward_history)
        ax5.plot(cum_rewards, color="green", alpha=0.8)
        ax5.set_title("AI Agent Cumulative Reward")
        ax5.set_xlabel("Time Step")
        ax5.set_ylabel("Cumulative Reward")
        ax5.grid(True, alpha=0.3)

        # Plot 6: Value Comparison Bar Chart
        ax6 = axes[1, 2]
        metrics = ["Final Value", "Fees Earned"]
        static_metrics = [
            s_values[-1],
            sum([abs(np.random.normal(0, 0.5)) for _ in range(len(s_values))]),
        ]
        ai_metrics = [
            a_values[-1],
            sum([abs(np.random.normal(0, 0.5)) for _ in range(len(a_values))]),
        ]
        x = np.arange(len(metrics))
        width = 0.35
        ax6.bar(x - width / 2, static_metrics, width, label="Static LP", color="blue", alpha=0.7)
        ax6.bar(x + width / 2, ai_metrics, width, label="AI Agent", color="green", alpha=0.7)
        ax6.set_title("Performance Comparison")
        ax6.set_xticks(x)
        ax6.set_xticklabels(metrics)
        ax6.legend()
        ax6.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig("aiamm_results.png", dpi=150, bbox_inches="tight")
        print("📊 Visualization saved to 'aiamm_results.png'")
        print()


print("Section 5 done")


Section 5 done


### Section 5: AIAMMSimulation - Experiment Engine
**Purpose**: Runs the complete A/B test comparing static vs AI-driven strategies

**Simulation Architecture**:
1. **run_static_strategy()**: Traditional LP behavior
   - Deposits 1000 of token A + 1000 of token B
   - Holds position passively (no rebalancing)
   - Earns random fees (simulating trading activity)
   - Accumulates impermanent loss as price moves

2. **run_ai_strategy()**: AI-optimized LP behavior
   - Same initial deposit (1000 + 1000)
   - Agent observes state and selects actions each step
   - Executes liquidity adjustments (increase/decrease deployment)
   - Trains via experience replay and Q-value updates
   - Aims to reduce IL while maximizing fees

3. **run_comparison()**: Orchestrates the full experiment
   - Runs N episodes of both strategies
   - Collects (values, IL, fees) for each episode
   - Computes statistics and comparisons
   - Generates 6 visualization plots
   - Prints detailed results table

**Results Metrics**:
- **Avg Final Portfolio Value**: Mean ending value across all episodes
- **Avg Impermanent Loss**: Mean IL% (lower is better for LP)
- **Avg Fees Earned**: Mean cumulative fees (higher is better)
- **Improvement %**: Percentage gain of AI over static
- **Std Dev**: Consistency/stability across episodes

**Visualization Outputs** (6 subplots):
1. Portfolio value trends over time
2. Impermanent loss comparison
3. Price history
4. Agent action distribution histogram
5. Cumulative reward learning curve
6. Final value & fees bar chart


In [7]:
# %% [markdown]
# ## Section 6: Main Execution
# Reference: `main()` - run this cell to execute the simulation end-to-end.

# %%


def main():
    print()
    print("╔══════════════════════════════════════════════════════════════════════╗")
    print("║                                                                      ║")
    print("║   AIAMM: Integrating AI Agents with Automated Market Makers          ║")
    print("║          for Smarter Liquidity Management                            ║")
    print("║                                                                      ║")
    print("║   Author: Franco Farrugia                                            ║")
    print("║   Research Area: Future Digital Technologies                         ║")
    print("║                                                                      ║")
    print("║   Algorithms: DQL | PPO | SAC (DQL demonstrated here)                ║")
    print("║   Components: AMM Simulation | RL Agent | Sentiment Analysis         ║")
    print("║                                                                      ║")
    print("╚══════════════════════════════════════════════════════════════════════╝")
    print()

    simulation = AIAMMSimulation(num_steps=500, num_episodes=50)
    results = simulation.run_comparison()
    print("Simulation complete.")
    print(results)


if __name__ == "__main__":
    main()



╔══════════════════════════════════════════════════════════════════════╗
║                                                                      ║
║   AIAMM: Integrating AI Agents with Automated Market Makers          ║
║          for Smarter Liquidity Management                            ║
║                                                                      ║
║   Author: Franco Farrugia                                            ║
║   Research Area: Future Digital Technologies                         ║
║                                                                      ║
║   Algorithms: DQL | PPO | SAC (DQL demonstrated here)                ║
║   Components: AMM Simulation | RL Agent | Sentiment Analysis         ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝

  AIAMM: AI Agents with AMMs for Smarter Liquidity Management

Running 50 simulation episodes...
Each episode: 500 time st

## Usage Guide & Interpretation

### How to Run
1. **Run All Cells in Order**: Execute cells 1-6 sequentially
   - Section 0: Imports & setup
   - Section 1-5: Define all classes
   - Section 6: Execute `main()` to run the simulation

2. **Adjust Parameters** (in Section 5):
   ```python
   # Current: 500 steps per episode, 50 episodes total
   simulation = AIAMMSimulation(num_steps=500, num_episodes=50)
   
   # For faster test runs:
   simulation = AIAMMSimulation(num_steps=50, num_episodes=2)
   ```
   - **num_steps**: Trading time steps per episode
   - **num_episodes**: How many independent runs to average

3. **Output Files**:
   - `aiamm_results.png`: 6-panel visualization saved to workspace

### Interpreting Results

**Portfolio Value**:
- Higher ending value = strategy performed better
- AI should show less volatility if learning effectively
- Watch for divergence between strategies over time

**Impermanent Loss**:
- Negative percentage (e.g., -5.7%) indicates loss vs HODL
- AI-driven should show lower IL% due to position adjustments
- IL reduction % shows improvement over static strategy

**Fees Earned**:
- Higher fees indicate more trading activity captured
- Static LP earns base fees; AI can amplify by deploying during high-volume periods
- Trade-off: More deployment = more fees but more exposure to price movement

**Action Distribution**:
- "Hold" dominance = conservative agent (needs more training)
- Balanced distribution = agent learning diverse strategies
- Patterns indicate when the agent increases/decreases liquidity

**Cumulative Reward**:
- Upward trend = agent improving over training
- Plateau = convergence to policy
- Negative trend = poor reward design or market conditions

### Key Findings to Look For
1. **Does AI outperform static LP?** - Check "Improvement %" in results
2. **On what metrics?** - Portfolio value? IL reduction? Fee collection?
3. **Is the agent learning?** - Does cumulative reward show upward trend?
4. **Is it stable?** - Do results vary much across episodes (check Std Dev)?
5. **When does it act?** - Does action distribution align with sentiment/volatility?

### Experiments to Try
- **Increase volatility**: Modify `AMMEnvironment.volatility` (default 0.02)
- **Larger price moves**: Increase `swap_amount` calculations in `simulate_market_activity()`
- **Different reward weights**: Adjust multipliers in `calculate_reward()` (-10, +5, +3)
- **More episodes**: Increase `num_episodes` for better statistical significance
- **Longer terms**: Increase `num_steps` to see long-term convergence
